In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
### Create the datapoints

from langsmith import Client
client=Client()

## Define the dataset these are your test data
dataset_name ="simple chat evaulations"
dataset=client.create_dataset(dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = [
    {
        "inputs": {
            "question": "What is Easy Build?"
        },
        "outputs": {
            "answer": "Easy Build is a construction management platform that helps contractors manage projects, budgets, and workflows."
        }
    },
    {
        "inputs": {
            "question": "What is the revenue goal?"
        },
        "outputs": {
            "answer": "The company's revenue goal for FY2025 is $10 million."
        }
    },
    {
        "inputs": {
            "question": "Who is the CEO of Easy Build?"
        },
        "outputs": {
            "answer": "The CEO of Easy Build is John Smith."
        }
    },
    {
        "inputs": {
            "question": "What industries does Easy Build serve?"
        },
        "outputs": {
            "answer": "Easy Build primarily serves construction companies, contractors, and infrastructure developers."
        }
    },
    {
        "inputs": {
            "question": "Where is Easy Build headquartered?"
        },
        "outputs": {
            "answer": "Easy Build is headquartered in Austin, Texas."
        }
    },
    {
        "inputs": {
            "question": "What are the key features of Easy Build?"
        },
        "outputs": {
            "answer": "Key features include project management, budgeting, document management, scheduling, and reporting."
        }
    }
]
)


{'example_ids': ['7322edaa-dfdc-44de-bc63-407df8f39155',
  'b0711b4d-a2cf-4e55-924e-e094cea6e921',
  'e20880e8-dc1d-4c93-ab8f-95b743f2143f',
  '72f44c6c-ce0d-42e6-8b4d-23d23c89aa3c',
  '4423dab2-cc45-4f9d-a0cb-c403bd91beb8',
  'c567c12b-2934-4364-847c-1c4e8ab800c5'],
 'count': 6,
 'as_of': '2026-07-28T13:32:05.854252198Z'}

In [ ]:
from src.models import build_llms
from langsmith import wrappers
from openai import OpenAI

grok_llm, openai_router, other_llm = build_llms()

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)
openai_router = wrappers.wrap_openai(openai_router)

local_llm = wrappers.wrap_openai(other_llm)

eval_instructions = "you are an expert professor specialized in grading students' answers to questions"

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""you are grading the following question:
{inputs['questions']}
here is the real answer:
{reference_outputs["answer"]}
you are grading the following predicated answer:
{outputs['response']}
Respond with Correct or incorrect:
Grade:
"""
    response = grok_llm.chat.completions.create(
        model="gemma-3-12b",
        temperature=0,
        messages=[
            {'role': 'system', 'content': eval_instructions},
            {'role': 'user', 'content': user_content},
        ]
    ).choices[0].message.content

    return response.strip().lower().startswith("correct")

AttributeError: 'ChatOpenAI' object has no attribute 'chat'